# a02 · per-seed attribution dataset -- generator for the between-draw check

Writes the per-seed attribution parquets `a02_shap_explain_n_stability.ipynb` §4 reads:
`seed_check/{version}_attributions_{split}_seed{seed}.parquet`, one file per `(version, split,
seed)` in `BETWEEN_DRAW_RUNS` x `SEEDS` below. Run this notebook once (top to bottom) before
opening §4 of that notebook; reruns are idempotent -- a seed already on disk is left alone unless
`FORCE_REGENERATE = True`.

**Kernel: the same analysis `.venv` as every other `notebook/real/` notebook.** This file does
NOT need a v2/v3 Jupyter kernel. It shells out to `config.python_bin(version)` -- that version's
OWN venv interpreter (`src/envs/v2/.venv`, `src/envs/v3/.venv`) -- to run
`src/attribution/attribute.py`, exactly the subprocess shape `src/pipeline/pipeline.py`'s
`_estimate()` already uses in this repo. SHAP needs the model *object*, which only unpickles
inside that version's own env (mutually incompatible library stacks per
`src/docs/ENV_MANAGEMENT.md`), so the analysis kernel never imports a model directly -- it just
launches a child process with a different interpreter. Nothing beyond that venv already existing
is required: no kernel registration, no extra config.

**Why a separate file from `a02_shap_explain_n_stability.ipynb`.** That notebook's own opening
cell promises "reads only already-saved output -- computes no new SHAP values"; folding
generation into it would break that promise and mix two concerns (produce vs. analyse) the rest
of this repo already keeps apart (compare `00_shap_attribution.ipynb`, which never opens a
pickle either). Keep `SEEDS` / `BETWEEN_DRAW_RUNS` / `SEED_SUBDIR` below identical to that
notebook's §4a copies -- both build the identical `seed_attribution_path()` and must resolve to
the same files.

In [ ]:
# §0 -- setup (analysis .venv kernel)
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config  # noqa: E402

print("ROOT =", ROOT)


In [ ]:
# §1 -- what to generate. Keep in sync with a02_shap_explain_n_stability.ipynb §4a.
SEEDS = [0, 1, 2, 3, 4]
BETWEEN_DRAW_RUNS = [("v2", "train"), ("v3", "train")]  # same worst-case pair §1 of that notebook uses
SEED_SUBDIR = "seed_check"

# attribute.py flags -- held at 00_SHAP.ipynb's own headline settings (interventional
# perturbation via --backend shap, background 500, explain sample N_EXPLAIN). Comparing seeds
# under different SHAP settings would put a measurement-comparability confound on top of the
# sampling question this notebook exists to answer, so none of these are varied per seed.
N_EXPLAIN = 5000       # 00_SHAP.ipynb §4's explain-sample size
BACKGROUND_N = 500     # 00_SHAP.ipynb's interventional background size
BACKEND = "shap"       # -> interventional perturbation, never "native"
FORCE_REGENERATE = False  # True re-runs attribute.py even for a seed already on disk


def seed_attribution_path(version: str, split: str, seed: int) -> Path:
    """Where the k-th independent-draw attribution parquet lives, alongside the canonical one
    (config.path's own template + SEED_SUBDIR), never a second registered config.py kind for a
    one-off validation exercise. Identical to a02_shap_explain_n_stability.ipynb's own copy."""
    canonical = config.path("attributions", version, "real", split=split)
    return canonical.parent / SEED_SUBDIR / f"{version}_attributions_{split}_seed{seed}.parquet"


In [ ]:
# §2 -- generate any missing seed attribution parquets, each in that version's OWN env
def _run(env_python: str, script: str, *args: str) -> subprocess.CompletedProcess:
    """Same subprocess shape as pipeline.py's `_run`/`_estimate` -- SHAP needs the model object,
    which only unpickles inside that version's own env, so this analysis-env notebook never
    imports it directly. Captures stdout/stderr instead of inheriting the OS file descriptors --
    a child process's direct writes to fd 1/2 do not reach this notebook's cell output, so
    `check=True` alone prints only CalledProcessError's bare repr (exit code, no message) and the
    actual attribute.py traceback stays invisible here even though the child process DID print
    one. Returning the CompletedProcess (never raising) lets the caller decide what to show."""
    return subprocess.run([env_python, str(ROOT / script), *args], cwd=ROOT,
                           capture_output=True, text=True)


for version, split in BETWEEN_DRAW_RUNS:
    env_py = str(config.python_bin(version))
    model_p = str(config.path("model", version, "real"))
    feats_p = str(config.path("processed_inputs", version, "real", split=split))
    registry_p = str(config.registry_path(version))   # attribute.py's fallback -- only consulted
                                                        # if the booster itself exposes no names
    print(f"\n=== generating seeds for {version}/{split} (env: {env_py}) ===")

    for seed in SEEDS:
        out_p = seed_attribution_path(version, split, seed)
        if out_p.is_file() and not FORCE_REGENERATE:
            print(f"  seed {seed}: already have {out_p.name}, skipping")
            continue
        out_p.parent.mkdir(parents=True, exist_ok=True)
        print(f"  seed {seed}: attribute.py -> {out_p}")
        result = _run(env_py, "src/attribution/attribute.py",
                      "--model", model_p, "--features", feats_p,
                      "--version", version, "--split", split,
                      "--rows", str(N_EXPLAIN), "--seed", str(seed),
                      "--background", str(BACKGROUND_N), "--backend", BACKEND,
                      "--features-json", registry_p,
                      "--out", str(out_p))
        if result.returncode != 0:
            print(f"  seed {seed}: FAILED (exit {result.returncode}) -- leaving this seed missing; "
                  f"a02_shap_explain_n_stability.ipynb §4b reports the pair as short rather "
                  f"than silently omitting it")
            if result.stdout.strip():
                print("  --- attribute.py stdout ---")
                print(result.stdout)
            if result.stderr.strip():
                print("  --- attribute.py stderr ---")
                print(result.stderr)
        elif result.stdout.strip():
            print(result.stdout)


## Next

Open `a02_shap_explain_n_stability.ipynb` and run §4a/§4b -- they read whatever this notebook
just wrote. Nothing else in that notebook depends on this file; §1-§3 read the canonical
(non-seed) attribution parquets `00_shap_attribution.ipynb`'s own prerequisite command produces,
unchanged.